In [6]:
import sqlite3
import os
import sys

def find_db():
    # Possible locations for bao.db
    locations = [
        '../../bao_server/bao_gnto.db',
    ]
    
    for loc in locations:
        if os.path.exists(loc):
            return loc
    return None

def inspect_db(db_path):
    print(f"Connecting to database at: {db_path}")
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Get all tables
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        
        if not tables:
            print("No tables found in the database.")
            return

        print(f"Found {len(tables)} tables.")
        
        for table in tables:
            table_name = table[0]
            print(f"\n{'='*40}")
            print(f"Table: {table_name}")
            print(f"{'='*40}")
            
            # Get column info
            cursor.execute(f"PRAGMA table_info({table_name})")
            columns = cursor.fetchall()
            col_names = [col[1] for col in columns]
            print(f"Columns: {col_names}")
            
            # Get row count
            cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cursor.fetchone()[0]
            print(f"Total rows: {count}")
            
            # Get sample data
            if count > 0:
                print("-" * 20)
                print("First 5 rows:")
                cursor.execute(f"SELECT * FROM {table_name} LIMIT 5")
                rows = cursor.fetchall()
                for row in rows:
                    print(row)
            else:
                print("(Empty table)")
                
        conn.close()
        
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
    except Exception as e:
        print(f"Error: {e}")

import csv
import os

def export_tables_to_csv(db_path, output_dir="exported_csvs"):
    """
    Exports all tables from the SQLite database at db_path to CSV files in output_dir.
    """
    print(f"Starting export from {db_path} to {output_dir}...")
    
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Get all table names
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        
        if not tables:
            print("No tables found in the database.")
            return

        for table in tables:
            table_name = table[0]
            csv_filename = f"{table_name}.csv"
            csv_path = os.path.join(output_dir, csv_filename)
            
            print(f"Exporting table '{table_name}'...")
            
            # Get column headers
            cursor.execute(f"PRAGMA table_info({table_name})")
            columns_info = cursor.fetchall()
            headers = [col[1] for col in columns_info]
            
            # Get all rows
            cursor.execute(f"SELECT * FROM {table_name}")
            rows = cursor.fetchall()
            
            # Write to CSV
            with open(csv_path, 'w', newline='', encoding='utf-8') as csv_file:
                csv_writer = csv.writer(csv_file)
                csv_writer.writerow(headers)  # Write header
                csv_writer.writerows(rows)    # Write data
                
            print(f"  -> Saved to {csv_path} ({len(rows)} rows)")
            
        print("\nAll tables exported successfully.")
        
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if conn:
            conn.close()


In [8]:
db_path = find_db()

if db_path:
    inspect_db(db_path)
    export_tables_to_csv(db_path)
else:
    print("Could not find 'bao.db' in current directory or '../bao_server/'")
    print("Please place 'bao.db' in this directory or update the script paths.")

Connecting to database at: ../../bao_server/bao_gnto.db
Found 3 tables.

Table: experience
Columns: ['id', 'pg_pid', 'plan', 'reward', 'iteration', 'episode']
Total rows: 1
--------------------
First 5 rows:
(1, 358235, '{"Plan": {"Node Type": "Other", "Node Type ID": "42", "Total Cost": 428263.671125, "Plan Rows": 1.0, "Plans": [{"Node Type": "Hash Join", "Node Type ID": "38", "Total Cost": 422555.271125, "Plan Rows": 2283356.0, "Plans": [{"Node Type": "Seq Scan", "Node Type ID": "19", "Relation Name": "movie_info", "Total Cost": 310999.35, "Plan Rows": 14900635.0}, {"Node Type": "Other", "Node Type ID": "47", "Total Cost": 67599.1625, "Plan Rows": 387402.0, "Plans": [{"Node Type": "Seq Scan", "Node Type ID": "19", "Relation Name": "title", "Total Cost": 67599.1625, "Plan Rows": 387402.0}]}]}]}, "Buffers": {"company_name_pkey": 883, "company_type_pkey": 2, "info_type_pkey": 2, "keyword_pkey": 503, "kind_type_pkey": 2, "movie_companies_pkey": 1, "company_id_movie_companies": 9076, "com